In [ ]:
from math import ceil, sqrt
from pathlib import Path
import shutil

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


In [ ]:
TASK_DIRNAME = "26Mar25-PyramidForcingView"
MANIFEST_FILENAME = "video_manifest.csv"
SUMMARY_FILENAME = "frame_export_summary.csv"
FRAME_STRIDE = 1
FRAME_FILENAME_TEMPLATE = "frame_{frame_idx:06d}.png"
SHEET_ROWS = 6
SHEET_COLS = 6
SHEET_DPI = 200
SHEET_FIGSIZE = (14, 14)
SHEET_FACE_COLOR = "white"
SHEET_OVERVIEW_MAX_COLS = 4
SKIP_EXISTING_VIDEO_DIRS = True


In [ ]:
def resolve_repo_root() -> Path:
    """Support running the notebook from repo root or from the notebook directory."""
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = resolve_repo_root()
DATA_DIR = REPO_ROOT / "data" / TASK_DIRNAME
FIG_DIR = REPO_ROOT / "figures" / TASK_DIRNAME
MANIFEST_PATH = DATA_DIR / MANIFEST_FILENAME
SUMMARY_PATH = DATA_DIR / SUMMARY_FILENAME
FIG_DIR.mkdir(parents=True, exist_ok=True)
{
    "repo_root": str(REPO_ROOT),
    "manifest_path": str(MANIFEST_PATH),
    "figure_root": str(FIG_DIR),
    "summary_path": str(SUMMARY_PATH),
    "frame_stride": FRAME_STRIDE,
}


In [ ]:
def load_manifest(manifest_path: str | Path) -> pd.DataFrame:
    """Load and validate the video manifest produced by the download notebook."""
    df = pd.read_csv(manifest_path)
    expected_columns = {
        "family",
        "duration",
        "repo_prefix",
        "repo_path",
        "source_filename",
        "video_index",
        "normalized_video_id",
        "local_video_path",
    }
    missing = expected_columns.difference(df.columns)
    if missing:
        raise ValueError(f"Missing columns from video manifest: {sorted(missing)}")

    df = df.sort_values(["family", "duration", "video_index"], kind="stable").reset_index(drop=True)
    return df


def chunked(items: list[Path], chunk_size: int) -> list[list[Path]]:
    return [items[index : index + chunk_size] for index in range(0, len(items), chunk_size)]


def render_image_grid(image_paths: list[Path], output_path: Path, rows: int, cols: int, title: str) -> None:
    """Render a contact sheet or overview grid from a list of PNG paths."""
    fig, axes = plt.subplots(rows, cols, figsize=SHEET_FIGSIZE, dpi=SHEET_DPI)
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    fig.patch.set_facecolor(SHEET_FACE_COLOR)
    fig.suptitle(title, fontsize=14)

    for axis, image_path in zip(axes, image_paths):
        image = plt.imread(image_path)
        axis.imshow(image)
        axis.axis("off")

    for axis in axes[len(image_paths) :]:
        axis.axis("off")

    fig.tight_layout()
    fig.savefig(output_path, dpi=SHEET_DPI, facecolor=SHEET_FACE_COLOR, bbox_inches="tight")
    plt.close(fig)


def render_sheet_pages(normalized_video_id: str, frame_paths: list[Path], video_dir: Path) -> list[Path]:
    """Split a long frame list into multiple paginated contact sheets."""
    page_size = SHEET_ROWS * SHEET_COLS
    pages = chunked(frame_paths, page_size)
    sheet_paths = []
    for page_index, page_frame_paths in enumerate(pages, start=1):
        sheet_path = video_dir / f"sheet_{page_index:03d}.png"
        render_image_grid(
            page_frame_paths,
            sheet_path,
            rows=SHEET_ROWS,
            cols=SHEET_COLS,
            title=f"{normalized_video_id} | page {page_index}/{len(pages)}",
        )
        sheet_paths.append(sheet_path)
    return sheet_paths


def render_sheet_overview(normalized_video_id: str, sheet_paths: list[Path], video_dir: Path) -> Path:
    """Create a single overview image summarizing all sheet pages for one video."""
    overview_path = video_dir / "sheet_overview.png"
    if len(sheet_paths) == 1:
        shutil.copy2(sheet_paths[0], overview_path)
        return overview_path

    overview_cols = min(SHEET_OVERVIEW_MAX_COLS, max(1, ceil(sqrt(len(sheet_paths)))))
    overview_rows = ceil(len(sheet_paths) / overview_cols)
    render_image_grid(
        sheet_paths,
        overview_path,
        rows=overview_rows,
        cols=overview_cols,
        title=f"{normalized_video_id} | sheet overview",
    )
    return overview_path


def export_video_frames(row: pd.Series) -> dict:
    """Export every selected frame and build paginated sheet images for one video."""
    normalized_video_id = row["normalized_video_id"]
    video_path = Path(row["local_video_path"])
    video_dir = FIG_DIR / normalized_video_id
    frames_dir = video_dir / "frames"
    overview_path = video_dir / "sheet_overview.png"

    if SKIP_EXISTING_VIDEO_DIRS and overview_path.exists() and frames_dir.exists():
        return {
            "normalized_video_id": normalized_video_id,
            "local_video_path": str(video_path),
            "decoded_frame_count": len(list(frames_dir.glob("frame_*.png"))),
            "exported_frame_count": len(list(frames_dir.glob("frame_*.png"))),
            "frame_stride": FRAME_STRIDE,
            "frame_dir": str(frames_dir),
            "sheet_count": len(list(video_dir.glob("sheet_[0-9][0-9][0-9].png"))),
            "overview_path": str(overview_path),
            "status": "skipped_existing",
        }

    if not video_path.exists():
        raise FileNotFoundError(f"Missing local video: {video_path}")

    video_dir.mkdir(parents=True, exist_ok=True)
    frames_dir.mkdir(parents=True, exist_ok=True)

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        capture.release()
        raise RuntimeError(f"Could not open video with OpenCV: {video_path}")

    decoded_frame_count = 0
    exported_frame_count = 0
    frame_paths = []

    while True:
        ok, frame_bgr = capture.read()
        if not ok:
            break

        decoded_frame_count += 1
        if (decoded_frame_count - 1) % FRAME_STRIDE != 0:
            continue

        frame_output_path = frames_dir / FRAME_FILENAME_TEMPLATE.format(frame_idx=decoded_frame_count)
        write_ok = cv2.imwrite(str(frame_output_path), frame_bgr)
        if not write_ok:
            capture.release()
            raise RuntimeError(f"OpenCV failed to write frame PNG: {frame_output_path}")

        exported_frame_count += 1
        frame_paths.append(frame_output_path)

    capture.release()

    if exported_frame_count == 0:
        raise RuntimeError(f"No frames were exported for {normalized_video_id}")

    sheet_paths = render_sheet_pages(normalized_video_id, frame_paths, video_dir)
    overview_path = render_sheet_overview(normalized_video_id, sheet_paths, video_dir)

    return {
        "normalized_video_id": normalized_video_id,
        "local_video_path": str(video_path),
        "decoded_frame_count": decoded_frame_count,
        "exported_frame_count": exported_frame_count,
        "frame_stride": FRAME_STRIDE,
        "frame_dir": str(frames_dir),
        "sheet_count": len(sheet_paths),
        "overview_path": str(overview_path),
        "status": "ok",
    }


In [ ]:
manifest_df = load_manifest(MANIFEST_PATH)
export_rows = []

for _, row in manifest_df.iterrows():
    try:
        export_rows.append(export_video_frames(row))
    except Exception as exc:
        export_rows.append(
            {
                "normalized_video_id": row["normalized_video_id"],
                "local_video_path": row["local_video_path"],
                "decoded_frame_count": 0,
                "exported_frame_count": 0,
                "frame_stride": FRAME_STRIDE,
                "frame_dir": str(FIG_DIR / row["normalized_video_id"] / "frames"),
                "sheet_count": 0,
                "overview_path": str(FIG_DIR / row["normalized_video_id"] / "sheet_overview.png"),
                "status": f"failed: {exc}",
            }
        )

summary_df = pd.DataFrame(export_rows)
summary_df.to_csv(SUMMARY_PATH, index=False)
display(summary_df)

failed_rows = summary_df[summary_df["status"].str.startswith("failed:")].reset_index(drop=True)
if not failed_rows.empty:
    display(failed_rows)
    failed_ids = ", ".join(failed_rows["normalized_video_id"].tolist())
    raise RuntimeError(f"Frame export failed for: {failed_ids}")

print(f"Saved frame export summary to {SUMMARY_PATH}")
